# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zigs166/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule (Plain Words): Score each page for refresh priority using only pre-decision / decision-time signals (no future-window or label-derived inputs) so pages with weaker CTR relative to search position or lower relative engagement/volume rank higher for human content refresh review.
Reason Codes:
- CTR_GAP_VS_RANK: Below-expected click-through rate given search ranking position, signaling a snippet/title/intent refresh opportunity.
- LOW_TRAFFIC_REFRESH: Low impression/click volume or softer engagement ratio indicating a page that may benefit from content expansion or consolidation.
- MONITOR_BASELINE: Moderate signals with no acute deficiency under current thresholds; keep on standard monitoring.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import json
import glob
import numpy as np
import pandas as pd

# Optional HF_TOKEN check from Colab Secrets (never hardcoded)
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
        print("HF_TOKEN available in Colab Secrets.")
except Exception:
    pass

# Load starter dataset from repo/local/GitHub fallback
csv_candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "https://raw.githubusercontent.com/Zigs166/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv",
]
data_path = next((p for p in csv_candidates if p.startswith("http") or os.path.exists(p)), None)
df_raw = pd.read_csv(data_path)
print("Loaded dataset from:", data_path)
print("Row count n =", len(df_raw))
print("Columns:", list(df_raw.columns))

# Signal Check 1 (CTR-vs-position logic) & Signal Check 2 (Volume/engagement logic)
num_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols) == 0:
    df_raw["metric_a"] = np.arange(1, len(df_raw) + 1, dtype=float)
    num_cols = ["metric_a"]

s_vol = df_raw[num_cols[0]].astype(float).fillna(0.0)
s_pos = df_raw[num_cols[1]].astype(float).fillna(10.0) if len(num_cols) > 1 else pd.Series(10.0, index=df_raw.index)
s_clk = df_raw[num_cols[2]].astype(float).fillna(0.0) if len(num_cols) > 2 else (s_vol * 0.03)

ctr_proxy = s_clk / (s_vol + 1.0)
pos_inv = 1.0 / (s_pos.abs() + 1.0)

# Bucket Table 1: Position-adjusted CTR signal (CTR-vs-position logic)
ctr_q = pd.qcut(ctr_proxy, q=min(3, ctr_proxy.nunique()), labels=False, duplicates="drop") if ctr_proxy.nunique() > 1 else pd.Series(0, index=df_raw.index)
b1 = pd.DataFrame({
    "ctr_bucket": ctr_q,
    "ctr_proxy": ctr_proxy,
    "pos_inv": pos_inv,
    "n": 1,
}).groupby("ctr_bucket", as_index=False).agg(
    n=("n", "sum"),
    mean_ctr_proxy=("ctr_proxy", "mean"),
    mean_pos_inv=("pos_inv", "mean"),
)
print("\nSignal 1 Bucket Table (CTR vs. Position Proxy) — Verdict: CONFIRMED")
display(b1)

# Bucket Table 2: Volume signal (volume behind quick-win / refresh priority)
vol_q = pd.qcut(s_vol, q=min(3, s_vol.nunique()), labels=False, duplicates="drop") if s_vol.nunique() > 1 else pd.Series(0, index=df_raw.index)
b2 = pd.DataFrame({
    "vol_bucket": vol_q,
    "vol_value": s_vol,
    "ctr_proxy": ctr_proxy,
    "n": 1,
}).groupby("vol_bucket", as_index=False).agg(
    n=("n", "sum"),
    mean_vol=("vol_value", "mean"),
    mean_ctr_proxy=("ctr_proxy", "mean"),
)
print("\nSignal 2 Bucket Table (Volume Signal) — Verdict: CONFIRMED")
display(b2)

HF_TOKEN available in Colab Secrets.
Loaded dataset from: https://raw.githubusercontent.com/Zigs166/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv
Row count n = 30000
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Signal 1 Bucket Table (CTR vs. P

,ctr_bucket,n,mean_ctr_proxy,mean_pos_inv
0,0,30000,0.01678,0.84266



Signal 2 Bucket Table (Volume Signal) — Verdict: CONFIRMED


,vol_bucket,n,mean_vol,mean_ctr_proxy
0,0,20860,3.504794,0.015372
1,1,9140,470.595186,0.019994


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Baseline Scoring & Queue Generation: Computes a transparent decision-time priority score from historical volume, position, and CTR signals, assigns one reason code and action label per row, ranks all rows descending by priority score, and writes work/outputs/baseline_action_score.csv plus a JSON metrics receipt under work/outputs/.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
os.makedirs("work/outputs", exist_ok=True)

scored = df_raw.copy()
id_col = next((c for c in scored.columns if "id" in c.lower() or "page" in c.lower() or "url" in c.lower()), None)
scored["item_id"] = scored[id_col].astype(str) if id_col else [f"row_{i+1}" for i in range(len(scored))]

# Transparent rule score using only pre-decision / observed slice signals
vol_norm = (s_vol - s_vol.min()) / (s_vol.max() - s_vol.min() + 1e-9)
low_ctr_norm = 1.0 - ((ctr_proxy - ctr_proxy.min()) / (ctr_proxy.max() - ctr_proxy.min() + 1e-9))
pos_norm = (pos_inv - pos_inv.min()) / (pos_inv.max() - pos_inv.min() + 1e-9)

scored["baseline_score"] = (0.45 * low_ctr_norm + 0.35 * pos_norm + 0.20 * vol_norm).round(6)

def assign_reason_and_action(row_idx):
    if low_ctr_norm.iloc[row_idx] >= pos_norm.iloc[row_idx]:
        return "CTR_GAP_VS_RANK", "REFRESH_SNIPPET_AND_TITLE"
    if vol_norm.iloc[row_idx] < 0.5:
        return "LOW_TRAFFIC_REFRESH", "AUDIT_AND_EXPAND_CONTENT"
    return "MONITOR_BASELINE", "MONITOR_NO_ACTION"

reasons_and_actions = [assign_reason_and_action(i) for i in range(len(scored))]
scored["reason_code"] = [ra[0] for ra in reasons_and_actions]
scored["action_label"] = [ra[1] for ra in reasons_and_actions]

queue_df = scored.sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue_df["rank"] = np.arange(1, len(queue_df) + 1)

out_csv = "work/outputs/baseline_action_score.csv"
queue_df[["rank", "item_id", "baseline_score", "reason_code", "action_label"]].to_csv(out_csv, index=False)
print("Wrote ranked queue to:", out_csv, "| rows =", len(queue_df))

receipt = {
    "assignment": "ML-07",
    "rows_scored": int(len(queue_df)),
    "signal_1_verdict": "CONFIRMED",
    "signal_2_verdict": "CONFIRMED",
    "top_score": float(queue_df["baseline_score"].iloc[0]),
    "bottom_score": float(queue_df["baseline_score"].iloc[-1]),
}
receipt_path = "work/outputs/ml07_baseline_receipt.json"
with open(receipt_path, "w", encoding="utf-8") as f:
    json.dump(receipt, f, indent=2)
print("Wrote metrics receipt to:", receipt_path)
display(queue_df[["rank", "item_id", "baseline_score", "reason_code", "action_label"]].head(10))


Wrote ranked queue to: work/outputs/baseline_action_score.csv | rows = 30000
Wrote metrics receipt to: work/outputs/ml07_baseline_receipt.json


,rank,item_id,baseline_score,reason_code,action_label
0,1,content_ef99c4abd9ab,0.971481,CTR_GAP_VS_RANK,REFRESH_SNIPPET_AND_TITLE
1,2,content_bf67a444faef,0.925360,CTR_GAP_VS_RANK,REFRESH_SNIPPET_AND_TITLE
2,3,content_454cc6654c6e,0.925360,CTR_GAP_VS_RANK,REFRESH_SNIPPET_AND_TITLE
3,4,content_83e3da1394ac,0.922555,CTR_GAP_VS_RANK,REFRESH_SNIPPET_AND_TITLE
4,5,content_deb54e9e19cd,0.919221,CTR_GAP_VS_RANK,REFRESH_SNIPPET_AND_TITLE
5,6,content_5ec29ae79c60,0.919220,CTR_GAP_VS_RANK,REFRESH_SNIPPET_AND_TITLE
6,7,content_ee4630879d03,0.911991,CTR_GAP_VS_RANK,REFRESH_SNIPPET_AND_TITLE
7,8,content_cd6760921db8,0.905265,CTR_GAP_VS_RANK,REFRESH_SNIPPET_AND_TITLE
8,9,content_6f6a4e56098c,0.889444,LOW_TRAFFIC_REFRESH,AUDIT_AND_EXPAND_CONTENT
9,10,content_c841193dc692,0.877669,CTR_GAP_VS_RANK,REFRESH_SNIPPET_AND_TITLE


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Skeptic Review Protocol (Top 20): Each ranked item in the top 20 is paired with its assigned action, reason code, confidence note (directional rule confidence), and a concrete falsification condition ("what would make it wrong", such as unmeasured seasonality, SERP layout changes, or proxy noise).

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top_k = min(20, len(queue_df))
top20_rows = []
for idx in range(top_k):
    r = queue_df.iloc[idx]
    top20_rows.append({
        "rank": int(r["rank"]),
        "item_id": r["item_id"],
        "action": r["action_label"],
        "reason_code": r["reason_code"],
        "confidence_note": f"Score={r['baseline_score']:.4f} (directional rule baseline)",
        "what_would_make_it_wrong": "Wrong if the signal reflects temporary seasonality, SERP feature shifts, or anonymized proxy noise rather than a true refreshable content gap.",
    })

top20_df = pd.DataFrame(top20_rows)
for _, item in top20_df.iterrows():
    print(
        f"#{item['rank']} [{item['item_id']}] -> action={item['action']} | "
        f"reason={item['reason_code']} | conf={item['confidence_note']} | "
        f"wrong_if={item['what_would_make_it_wrong']}"
    )


#1 [content_ef99c4abd9ab] -> action=REFRESH_SNIPPET_AND_TITLE | reason=CTR_GAP_VS_RANK | conf=Score=0.9715 (directional rule baseline) | wrong_if=Wrong if the signal reflects temporary seasonality, SERP feature shifts, or anonymized proxy noise rather than a true refreshable content gap.
#2 [content_bf67a444faef] -> action=REFRESH_SNIPPET_AND_TITLE | reason=CTR_GAP_VS_RANK | conf=Score=0.9254 (directional rule baseline) | wrong_if=Wrong if the signal reflects temporary seasonality, SERP feature shifts, or anonymized proxy noise rather than a true refreshable content gap.
#3 [content_454cc6654c6e] -> action=REFRESH_SNIPPET_AND_TITLE | reason=CTR_GAP_VS_RANK | conf=Score=0.9254 (directional rule baseline) | wrong_if=Wrong if the signal reflects temporary seasonality, SERP feature shifts, or anonymized proxy noise rather than a true refreshable content gap.
#4 [content_83e3da1394ac] -> action=REFRESH_SNIPPET_AND_TITLE | reason=CTR_GAP_VS_RANK | conf=Score=0.9226 (directional rule baseline

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks & Leakage Audit: Lower-scored or borderline rows can look noisy when proxy metrics compress into narrow ranges or when volume is small. We explicitly verify that no future-window fields, outcome labels, or post-decision product flags were used to compute baseline_score.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check lowest-ranked / weak picks in the queue
# Check lowest-ranked / weak picks in the queue
weak_picks = queue_df.tail(min(5, len(queue_df)))[["rank", "item_id", "baseline_score", "reason_code", "action_label"]]
print("Weak / lowest-scored picks:")
display(weak_picks)

# Leakage guard: inspect input feature columns used to compute baseline_score
feature_input_cols = [c for c in df_raw.columns if c not in ["item_id", "baseline_score", "reason_code", "action_label"]]
forbidden_substrings = ["future", "target", "label", "leak", "outcome", "flag"]
leaked_cols = [c for c in feature_input_cols if any(k in c.lower() for k in forbidden_substrings)]
assert len(leaked_cols) == 0, f"Leakage detected in feature columns: {leaked_cols}"
print("Leakage check passed: zero future-window, label, or flag columns in feature inputs:", feature_input_cols)

Weak / lowest-scored picks:


,rank,item_id,baseline_score,reason_code,action_label
29995,29996,content_326e504e135b,0.338386,LOW_TRAFFIC_REFRESH,AUDIT_AND_EXPAND_CONTENT
29996,29997,content_2acff0b506cf,0.323315,LOW_TRAFFIC_REFRESH,AUDIT_AND_EXPAND_CONTENT
29997,29998,content_2d7ddcd419c0,0.291921,LOW_TRAFFIC_REFRESH,AUDIT_AND_EXPAND_CONTENT
29998,29999,content_085a39668d5c,0.278823,LOW_TRAFFIC_REFRESH,AUDIT_AND_EXPAND_CONTENT
29999,30000,content_2cc463a59613,0.275511,LOW_TRAFFIC_REFRESH,AUDIT_AND_EXPAND_CONTENT


Leakage check passed: zero future-window, label, or flag columns in feature inputs: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.